In [10]:
import os; imimport os
import time
import numpy as np
from qiskit_algorithms import QAOAport time; import numpy as npfrom qiskit_algorithms import QAOA
from qiskit_algorithms.optimizers import COBYLA

warnings.filterwarnings('ignore')

dirs = {'tables': 'results/tables', 'figures': 'results/figures'}
for path in dirs.values():
    os.makedirs(path, exist_ok=True)

# ==========================================
# 3. PROBLEM DEFINITION (MAX-CUT GRAPH)
# ==========================================
G = nx.Graph()
G.add_edges_from([(0, 1), (1, 2), (2, 3), (3, 4), (4, 0), (0, 2)])

maxcut = Maxcut(G)
qp = maxcut.to_quadratic_program()
observable, offset = qp.to_ising()

# ==========================================
# 4. PROCESSOR CONFIGURATION
# ==========================================
processors = {
    'Processor_A': FakeManilaV2(),
    'Processor_B': FakeNairobiV2()
}

# ==========================================
# 5. BENCHMARKING FUNCTION (QAOA EXECUTION)
# ==========================================
def benchmark_qaoa(processor_name, backend, observable):
    print(f"Benchmarking {processor_name}...")

    simulator = AerSimulator.from_backend(backend)

    sampler = AerSampler(
        backend_options={
            "noise_model": simulator.noise_model,
            "coupling_map": simulator.coupling_map,
            "basis_gates": simulator.operation_names
        },
        run_options={"shots": 1024}
    )

    optimizer = COBYLA(maxiter=50)
    qaoa = QAOA(sampler, optimizer, reps=2)

    start_time = time.time()
    result = qaoa.compute_minimum_eigenvalue(observable)
    execution_time = time.time() - start_time

    best_bitstring = None
    max_prob = 0

    for state_int, prob in result.eigenstate.items():
        bitstring = format(state_int, f'0{observable.num_qubits}b')
        if prob > max_prob:
            max_prob = prob
            best_bitstring = bitstring

    x = [int(bit) for bit in best_bitstring][::-1]
    cut_value = maxcut.objective_value(x)

    return {
        'Processor': processor_name,
        'Execution_Time_sec': round(execution_time, 2),
        'Max_Cut_Value': cut_value
    }

# ==========================================
# 6. RUN EXPERIMENTS & EXPORT DATA
# ==========================================
results_data = []
for name, backend in processors.items():
    results_data.append(benchmark_qaoa(name, backend, observable))

df_results = pd.DataFrame(results_data)
csv_path = f"{dirs['tables']}/PS_O1_benchmark_results.csv"
df_results.to_csv(csv_path, index=False)

print("\n==========================================")
print("BENCHMARK COMPLETE")
print("==========================================")
print(df_results.to_string(index=False))


SyntaxError: invalid syntax (3623853730.py, line 1)